# Window Functions


## OVER()

In [ ]:
The Problem With GROUP BY

Consider:

employees
| emp_id | department | salary |
| ------ | ---------- | ------ |
| 1      | IT         | 90000  |
| 2      | IT         | 70000  |
| 3      | IT         | 50000  |
| 4      | HR         | 60000  |
| 5      | HR         | 40000  |

Suppose I ask:

Show every employee and also show the average salary of their department.

Most beginners try:

SELECT
    department,
    AVG(salary)
FROM employees
GROUP BY department;

Result:
| department | avg_salary |
| ---------- | ---------- |
| IT         | 70000      |
| HR         | 50000      |

But notice:

❌ We lost individual employees.

What We Want
| emp_id | department | salary | dept_avg_salary |
| ------ | ---------- | ------ | --------------- |
| 1      | IT         | 90000  | 70000           |
| 2      | IT         | 70000  | 70000           |
| 3      | IT         | 50000  | 70000           |
| 4      | HR         | 60000  | 50000           |
| 5      | HR         | 40000  | 50000           |

We want:

Individual rows preserved
Aggregate information attached

This is exactly what Window Functions do.

In [ ]:
First Window Function
AVG(salary) OVER (
    PARTITION BY department
)

Query:

SELECT
    emp_id,
    department,
    salary,
    AVG(salary) OVER (
        PARTITION BY department
    ) AS dept_avg_salary
FROM employees;

Result:
| emp_id | department | salary | dept_avg_salary |
| ------ | ---------- | ------ | --------------- |
| 1      | IT         | 90000  | 70000           |
| 2      | IT         | 70000  | 70000           |
| 3      | IT         | 50000  | 70000           |
| 4      | HR         | 60000  | 50000           |
| 5      | HR         | 40000  | 50000           |

Mental Model

Think:

GROUP BY

= Collapse rows

OVER()

= Keep rows and calculate something across them

This distinction is crucial.

## ROW_NUMBER()

In [ ]:
This is one of the most important SQL functions for Data Analysts.

What does ROW_NUMBER() do?

It assigns a unique sequential number to each row.

Example:
| emp_id | salary |
| ------ | ------ |
| 1      | 100000 |
| 2      | 50000  |
| 3      | 70000  |

Query:

SELECT
    emp_id,
    salary,
    ROW_NUMBER() OVER (
        ORDER BY salary DESC
    ) AS row_num
FROM employees;

Result:
| emp_id | salary | row_num |
| ------ | ------ | ------- |
| 1      | 100000 | 1       |
| 3      | 70000  | 2       |
| 2      | 50000  | 3       |


In [ ]:
Mental Model
ORDER BY salary DESC

Sorts:
| salary |
| ------ |
| 100000 |
| 70000  |
| 50000  |

Then:

ROW_NUMBER()

assigns:
1
2
3


In [ ]:
Why Analysts Use It
Find Top Earners
SELECT *
FROM (
    SELECT
        emp_id,
        salary,
        ROW_NUMBER() OVER (
            ORDER BY salary DESC
        ) AS rn
    FROM employees
) t
WHERE rn <= 3;

Top 3 highest-paid employees.

Very common interview question.

## PARTITION BY + ROW_NUMBER()

In [ ]:
Now it gets interesting.

Suppose:
| emp_id | department | salary |
| ------ | ---------- | ------ |
| 1      | IT         | 100000 |
| 2      | IT         | 50000  |
| 3      | IT         | 70000  |
| 4      | HR         | 60000  |
| 5      | HR         | 40000  |

Query:

SELECT
    emp_id,
    department,
    salary,
    ROW_NUMBER() OVER (
        PARTITION BY department
        ORDER BY salary DESC
    ) AS rn
FROM employees;

IT Department
Sort salaries:

| emp_id | salary |
| ------ | ------ |
| 1      | 100000 |
| 3      | 70000  |
| 2      | 50000  |

Assign:
1
2
3

HR Department
Sort salaries:

| emp_id | salary |
| ------ | ------ |
| 4      | 60000  |
| 5      | 40000  |

Assign:
1
2

Final Result
| emp_id | department | salary | rn |
| ------ | ---------- | ------ | -- |
| 1      | IT         | 100000 | 1  |
| 3      | IT         | 70000  | 2  |
| 2      | IT         | 50000  | 3  |
| 4      | HR         | 60000  | 1  |
| 5      | HR         | 40000  | 2  |

Notice:
The numbering restarts for each department.
That's what PARTITION BY does.


## RANK()

In [ ]:
Consider:
| emp_id | salary |
| ------ | ------ |
| 1      | 100000 |
| 2      | 100000 |
| 3      | 70000  |

Query:

SELECT
    emp_id,
    salary,
    ROW_NUMBER() OVER (
        ORDER BY salary DESC
    ) AS rn
FROM employees;

Result:
| emp_id | salary | rn |
| ------ | ------ | -- |
| 1      | 100000 | 1  |
| 2      | 100000 | 2  |
| 3      | 70000  | 3  |

Notice:

Both employees have the same salary.

Yet one got rank 1 and the other got rank 2.

Why?

Because ROW_NUMBER() doesn't care about ties.

It simply says:

"You are first."

"You are second."

"You are third."

Even if values are identical.



In [ ]:
Enter RANK()

Same data:
| emp_id | salary |
| ------ | ------ |
| 1      | 100000 |
| 2      | 100000 |
| 3      | 70000  |

Query:

SELECT
    emp_id,
    salary,
    RANK() OVER (
        ORDER BY salary DESC
    ) AS rnk
FROM employees;

Result:
| emp_id | salary | rnk |
| ------ | ------ | --- |
| 1      | 100000 | 1   |
| 2      | 100000 | 1   |
| 3      | 70000  | 3   |

Notice:
1
1
3

Rank 2 is skipped.
This is called a gap rank.

## Dense Rank()

In [ ]:
SELECT
    emp_id,
    salary,
    DENSE_RANK() OVER (
        ORDER BY salary DESC
    ) AS drnk
FROM employees;

Result:
| emp_id | salary | drnk |
| ------ | ------ | ---- |
| 1      | 100000 | 1    |
| 2      | 100000 | 1    |
| 3      | 70000  | 2    |

Notice:
1
1
2

No gap.

## LAG() — Compare With Previous Row

In [ ]:
This is one of the most useful SQL functions for analytics.

Imagine you have monthly revenue:
| month | revenue |
| ----- | ------- |
| Jan   | 1000    |
| Feb   | 1200    |
| Mar   | 900     |
| Apr   | 1500    |

Management asks:

"How much did revenue change compared to last month?"

To answer this, we need access to the previous row.

In [ ]:
Enter LAG()

LAG(column_name)

Think:

"Give me the value from the previous row."

Example
SELECT
    month,
    revenue,
    LAG(revenue) OVER (
        ORDER BY month
    ) AS previous_revenue
FROM sales;

Result:
| month | revenue | previous_revenue |
| ----- | ------- | ---------------- |
| Jan   | 1000    | NULL             |
| Feb   | 1200    | 1000             |
| Mar   | 900     | 1200             |
| Apr   | 1500    | 900              |

Why is Jan NULL?

Because there's no previous month.


In [ ]:
Real Analyst Use Case #1

Calculate month-over-month change.

SELECT
    month,
    revenue,
    LAG(revenue) OVER (
        ORDER BY month
    ) AS previous_revenue,
    revenue -
    LAG(revenue) OVER (
        ORDER BY month
    ) AS revenue_change
FROM sales;

Result:
| month | revenue | previous_revenue | revenue_change |
| ----- | ------- | ---------------- | -------------- |
| Jan   | 1000    | NULL             | NULL           |
| Feb   | 1200    | 1000             | 200            |
| Mar   | 900     | 1200             | -300           |
| Apr   | 1500    | 900              | 600            |

Now you can immediately see growth and decline.


In [ ]:
Real Analyst Use Case #2

Percentage Growth

Formula:

(Current - Previous) / Previous * 100

PostgreSQL:

SELECT
    month,
    revenue,
    ROUND(
        (
            revenue -
            LAG(revenue) OVER (
                ORDER BY month
            )
        ) * 100.0
        /
        LAG(revenue) OVER (
            ORDER BY month
        ),
        2
    ) AS growth_percent
FROM sales;

Result:
| month | growth_percent |
| ----- | -------------- |
| Jan   | NULL           |
| Feb   | 20.00          |
| Mar   | -25.00         |
| Apr   | 66.67          |

This is exactly the type of KPI analysts build.

## LAG With PARTITION BY

In [ ]:
Suppose:
| employee | department | month | sales |
| -------- | ---------- | ----- | ----- |
| A        | IT         | Jan   | 100   |
| A        | IT         | Feb   | 120   |
| B        | HR         | Jan   | 50    |
| B        | HR         | Feb   | 70    |

We want previous sales within each employee.

SELECT
    employee,
    month,
    sales,
    LAG(sales) OVER (
        PARTITION BY employee
        ORDER BY month
    ) AS previous_sales
FROM performance;

The comparison restarts for each employee.

Very common pattern.

In [ ]:
Important Feature

You can go back multiple rows.

Previous 2 rows
LAG(revenue, 2)

Example:
| month | revenue | lag_2 |
| ----- | ------- | ----- |
| Jan   | 1000    | NULL  |
| Feb   | 1200    | NULL  |
| Mar   | 900     | 1000  |
| Apr   | 1500    | 1200  |

Meaning:

"Show me the value from 2 rows ago."

## Lead() - Compare With Next Row

In [ ]:
Think:

"Give me the value from the next row."

Example

Table:
| month | revenue |
| ----- | ------- |
| Jan   | 1000    |
| Feb   | 1200    |
| Mar   | 900     |
| Apr   | 1500    |

Query:

SELECT
    month,
    revenue,
    LEAD(revenue) OVER (
        ORDER BY month
    ) AS next_revenue
FROM sales;

Result:
| month | revenue | next_revenue |
| ----- | ------- | ------------ |
| Jan   | 1000    | 1200         |
| Feb   | 1200    | 900          |
| Mar   | 900     | 1500         |
| Apr   | 1500    | NULL         |

Notice:

April gets NULL because there is no next month.


In [ ]:
Mental Model

Imagine the revenue column:

1000
1200
900
1500

LAG() shifts values upward:

NULL
1000
1200
900

LEAD() shifts values downward:

1200
900
1500
NULL

This mental image helps a lot.

In [ ]:
Real Analyst Use Case #1

Find customers whose next purchase is known.

Imagine:
| customer_id | order_date |
| ----------- | ---------- |
| 101         | 2025-01-10 |
| 101         | 2025-02-15 |
| 101         | 2025-04-20 |

Query:

SELECT
    customer_id,
    order_date,
    LEAD(order_date) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS next_order_date
FROM orders;

Result:
| customer_id | order_date | next_order_date |
| ----------- | ---------- | --------------- |
| 101         | Jan 10     | Feb 15          |
| 101         | Feb 15     | Apr 20          |
| 101         | Apr 20     | NULL            |

Very useful for retention analysis.

In [ ]:
Real Analyst Use Case #2

Days Until Next Order

PostgreSQL:

SELECT
    customer_id,
    order_date,
    LEAD(order_date) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) - order_date AS days_until_next_order
FROM orders;

This tells you how many days customers take before buying again.

Companies care a lot about this metric.

In [ ]:
LEAD with Offset

Next 2 rows:

LEAD(revenue, 2)

Example:
| month | revenue | lead_2 |
| ----- | ------- | ------ |
| Jan   | 1000    | 900    |
| Feb   | 1200    | 1500   |
| Mar   | 900     | NULL   |
| Apr   | 1500    | NULL   |

LAG vs LEAD
Function	Meaning
LAG()	Previous row
LEAD()	Next row

Examples:

LAG(revenue)
Current month vs Previous month
LEAD(revenue)
Current month vs Next month